# Install Required Libraries

In [26]:
pip install sklearn-crfsuite scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Import Libraries

In [27]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics

# Check Working Directory

In [28]:
import os

print(os.getcwd())

C:\Users\user name\Desktop\CRF_POS_Assignment


# Load Dataset

In [29]:
file_path = "mypos-ver.3.0.txt"

with open(file_path, "r", encoding="utf-8") as f:
    data = f.readlines()

print("Number of sentences:", len(data))

Number of sentences: 43196


# Check Dataset Sample

In [30]:
for i in range(5):
    print("Sentence", i+1)
    print(data[i])
    print("-"*50)

Sentence 1
ဒီ/adj ဆေး/n က/ppm ၁၀၀/num ရာခိုင်နှုန်း/n ဆေးဘက်ဝင်/adj အပင်/n များ/part မှ/ppm ဖော်စပ်/v ထား/part တာ/part ဖြစ်/v တယ်/ppm ။/punc

--------------------------------------------------
Sentence 2
အသစ်/n ဝယ်/v ထား/part တဲ့/part ဆွယ်တာ/n က/ppm အသီး/n ထ/v နေ/part ပါ/part ပေါ့/part ။/punc

--------------------------------------------------
Sentence 3
မ/part ကျန်းမာ/v လျှင်/conj နတ်/n|ဆရာ/n ထံ/ppm မေးမြန်း/v ၍/conj သက်ဆိုင်ရာ/n နတ်/n တို့/part အား/ppm ပူဇော်ပသ/v ရ/part သည်/ppm ။/punc

--------------------------------------------------
Sentence 4
ပေဟိုင်/n|ဥယျာဉ်/n ။/punc

--------------------------------------------------
Sentence 5
နဝမ/adj အိပ်မက်/n ကောသလ/n|မင်း/n|အိပ်မက်/n ၉/num နက်ရှိုင်း/adj ကျယ်ဝန်း/adj သော/part ရေကန်/n ကြီး/adj တစ်/tn ခု/part တွင်/ppm သတ္တဝါ/n တို့/part ဆင်း/v ၍/conj ရေသောက်/v ကြ/part ၏/ppm ။/punc

--------------------------------------------------


# Convert Corpus into Word + POS Format

In [31]:
def read_pos_corpus(lines):

    sentences = []

    for line in lines:

        tokens = line.strip().split()

        sentence = []

        for token in tokens:

            if "/" in token:

                word, tag = token.rsplit("/",1)

                sentence.append((word, tag))


        if sentence:
            sentences.append(sentence)

    return sentences

In [32]:
pos_sentences = read_pos_corpus(data)

print("Total sentences:", len(pos_sentences))

print(pos_sentences[0])

Total sentences: 43196
[('ဒီ', 'adj'), ('ဆေး', 'n'), ('က', 'ppm'), ('၁၀၀', 'num'), ('ရာခိုင်နှုန်း', 'n'), ('ဆေးဘက်ဝင်', 'adj'), ('အပင်', 'n'), ('များ', 'part'), ('မှ', 'ppm'), ('ဖော်စပ်', 'v'), ('ထား', 'part'), ('တာ', 'part'), ('ဖြစ်', 'v'), ('တယ်', 'ppm'), ('။', 'punc')]


# Feature Extraction

In [42]:
def word_features(sentence, i):

    word = sentence[i][0]

    features = {
        'word': word,
        'bias': 1.0,
        'prefix': word[:2],
        'suffix': word[-2:],
        'length': len(word)
    }


    if i > 0:
        features['prev_word'] = sentence[i-1][0]
    else:
        features['BOS'] = True


    if i < len(sentence)-1:
        features['next_word'] = sentence[i+1][0]
    else:
        features['EOS'] = True


    return features

# Prepare CRF Data

In [43]:
def prepare_crf_data(sentences):

    X=[]
    y=[]


    for sentence in sentences:

        X.append(
            [
                word_features(sentence,i)
                for i in range(len(sentence))
            ]
        )


        y.append(
            [
                tag
                for word,tag in sentence
            ]
        )


    return X,y

In [56]:
X,y = prepare_crf_data(pos_sentences)


print("Feature extraction completed!")
print("Number of sentences prepared:", len(X))

Feature extraction completed!
Number of sentences prepared: 43196


# Split Training and Testing Data

In [45]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("Training sentences:", len(X_train))
print("Testing sentences:", len(X_test))

Training sentences: 34556
Testing sentences: 8640


# Build CRF Model

In [46]:
import sklearn_crfsuite

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    max_iterations=100,
    all_possible_transitions=True
)


print("Training CRF POS Tagger Model...")


crf.fit(
    X_train,
    y_train
)


print("Model training completed successfully!")

Training CRF POS Tagger Model...
Model training completed successfully!


# Prediction

In [47]:
y_pred = crf.predict(X_test)


print("Prediction completed!")

Prediction completed!


# Example Prediction

In [48]:
print("Actual POS tags:")
print(y_test[0])


print("\nPredicted POS tags:")
print(y_pred[0])

Actual POS tags:
['tn', 'punc']

Predicted POS tags:
['tn', 'punc']


# Evaluation

In [49]:
from sklearn_crfsuite import metrics
accuracy = metrics.flat_accuracy_score(
    y_test,
    y_pred
)


print("POS Tagging Accuracy:", accuracy)

print(
    "Accuracy Percentage:",
    round(accuracy*100,2),
    "%"
)

POS Tagging Accuracy: 0.8824663527492492
Accuracy Percentage: 88.25 %


In [50]:
report = metrics.flat_classification_report(
    y_test,
    y_pred,
    zero_division=0
)


print(report)

              precision    recall  f1-score   support

         abb       0.00      0.00      0.00        72
         adj       0.70      0.59      0.64      3214
         adv       0.67      0.48      0.56      2142
        conj       0.84      0.82      0.83      3623
          fw       0.32      0.27      0.29       644
         int       0.93      0.36      0.52       137
           n       0.80      0.88      0.84     21040
         num       0.91      0.74      0.81      1174
        part       0.90      0.91      0.91     26634
         ppm       0.96      0.97      0.96     17509
        pron       0.91      0.88      0.89      4046
        punc       1.00      0.99      1.00     10859
          sb       0.00      0.00      0.00        45
          tn       0.90      0.82      0.86      1174
           v       0.87      0.84      0.86     15571

    accuracy                           0.88    107884
   macro avg       0.71      0.64      0.66    107884
weighted avg       0.88   

# POS Tagging Testing

In [52]:
test_file = "mypos-ver.3.0.shuf.notag.nopunc.txt"


with open(test_file,"r",encoding="utf-8") as f:
    raw_sentences = f.readlines()


print(raw_sentences[0])

sentence = raw_sentences[0].strip()


words = sentence.split()


print(words)

၁၉၆၂ ခုနှစ် ခန့်မှန်း သန်းခေါင်စာရင်း အရ လူဦးရေ ၁၁၅၉၃၁ ယောက် ရှိ သည်

['၁၉၆၂', 'ခုနှစ်', 'ခန့်မှန်း', 'သန်းခေါင်စာရင်း', 'အရ', 'လူဦးရေ', '၁၁၅၉၃၁', 'ယောက်', 'ရှိ', 'သည်']


In [54]:
test_sentence = [
    (word,"")
    for word in words
]


test_features = [
    word_features(test_sentence,i)
    for i in range(len(test_sentence))
]

In [55]:
prediction = crf.predict(
    [test_features]
)


for word,tag in zip(words,prediction[0]):

    print(word,"--->",tag)

၁၉၆၂ ---> num
ခုနှစ် ---> n
ခန့်မှန်း ---> n
သန်းခေါင်စာရင်း ---> n
အရ ---> ppm
လူဦးရေ ---> n
၁၁၅၉၃၁ ---> n
ယောက် ---> part
ရှိ ---> v
သည် ---> ppm
